<a href="https://colab.research.google.com/github/Darshanbreddy/LLM/blob/main/ADK_Learning_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [2]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries for our entire adventure ---
import os
import re
import asyncio
from IPython.display import display, Markdown
import google.generativeai as genai
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part
from getpass import getpass

print("✅ All libraries are ready to go!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ All libraries are ready to go!


In [4]:
# --- Securely Configure Your API Key ---

# Prompt the user for their API key securely
api_key = getpass('Enter your Google API Key: ')

# Configure the generative AI library with the provided key
genai.configure(api_key=api_key)

# Set the API key as an environment variable for ADK to use
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully! Let the fun begin.")

Enter your Google API Key: ··········
✅ API Key configured successfully! Let the fun begin.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [5]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [6]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [9]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Mysore,KA. Keep it Expensive!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Mysore,KA. Keep it Expensive!'

🚀 Running query for agent: 'day_trip_agent' in session: 'fb1a6e25-47d5-40cf-bec4-bef626487e48'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's an expensive, relaxing, and artsy day trip near Mysore, Karnataka, designed to immerse you in culture and luxury:

---

## **A Lavish Canvas: Relaxing & Artsy Day Trip near Mysore**

This exclusive itinerary focuses on high-end art experiences, serene historical sites, and exquisite dining, ensuring a truly indulgent and relaxing day. Private luxury car transfers are recommended throughout the day for ultimate comfort and convenience.

### **Morning: Royal Artistry & Hands-On Creativity**

*   **9:30 AM - 11:30 AM: Immerse in Royal Art at Jaganmohan Palace Art Gallery**
    Begin your day with a visit to the **Sri Jayachamarajendra Art Gallery**, housed within the magnificent Jaganmohan Palace. This esteeme

Here's an expensive, relaxing, and artsy day trip near Mysore, Karnataka, designed to immerse you in culture and luxury:

---

## **A Lavish Canvas: Relaxing & Artsy Day Trip near Mysore**

This exclusive itinerary focuses on high-end art experiences, serene historical sites, and exquisite dining, ensuring a truly indulgent and relaxing day. Private luxury car transfers are recommended throughout the day for ultimate comfort and convenience.

### **Morning: Royal Artistry & Hands-On Creativity**

*   **9:30 AM - 11:30 AM: Immerse in Royal Art at Jaganmohan Palace Art Gallery**
    Begin your day with a visit to the **Sri Jayachamarajendra Art Gallery**, housed within the magnificent Jaganmohan Palace. This esteemed gallery boasts a collection of over 2,000 paintings, showcasing diverse schools of Indian art, including traditional Mysore paintings with their intricate detailing and works by the celebrated Raja Ravi Varma. Take your time to appreciate the opulent surroundings and the historical significance of the artworks. The gallery is open from 8:30 AM to 5:00 PM daily.
*   **11:45 AM - 1:00 PM: Private Mysore Painting Workshop**
    Engage your own artistic spirit with a private workshop focused on traditional Mysore painting or another local art form. Establishments like **Kalayana** or **The Hobby Place** in Mysore offer personalized guidance from experienced instructors, allowing you to learn foundational techniques and create your own masterpiece in a calm and beautiful ambiance. This hands-on experience provides a unique and memorable souvenir.

### **Lunch: Gourmet Indulgence**

*   **1:30 PM - 2:45 PM: Fine Dining at a Heritage Hotel**
    Savor a lavish lunch at one of Mysore's premier dining establishments. Opt for **Tiger Trail at Royal Orchid Metropole Hotel**, known for its elegant ambiance and exquisite North Indian and Mughlai cuisine, perfect for a special occasion. Alternatively, **La Uppu at Grand Mercure Mysore** offers an upscale, contemporary South Indian and multi-cuisine buffet experience.

### **Afternoon: Serene History & Ancient Art**

*   **3:15 PM - 6:00 PM: Explore the Buried Temples of Talakadu**
    Embark on a scenic drive (approximately 1.5 hours) to **Talakadu**, often referred to as the "city beneath the sands." This ancient site, about 45-50 km southeast of Mysore, is known for its cluster of historically significant temples, many of which were once buried under sand dunes. With a private guide, explore the **Vaidyanatheshwara Temple**, noted for its majestic architecture, beautiful ornate carvings, and artistic sculptures. The serene, somewhat mystical atmosphere of Talakadu offers a truly relaxing and culturally rich experience away from the city's bustle.

### **Evening: Relaxation & Exquisite Dinner**

*   **6:30 PM - 8:00 PM: Rejuvenating Spa Treatment**
    Return to Mysore and unwind with a luxurious spa treatment. **Silent Shores Resort & Spa** or **The Windflower Resorts and Spa** are renowned for their lavish accommodations and world-class spa services, offering a tranquil environment for rejuvenation. Indulge in a massage or a wellness therapy to fully relax after a day of exploration.
*   **8:15 PM onwards: Sophisticated European Dinner**
    Conclude your day with a sophisticated European fine dining experience at **Mezzaluna**. Enjoy their carefully crafted steaks, pastas, and Mediterranean dishes in a warm and stylish setting, accompanied by a fine selection of wines. For an equally refined experience, **Spring at Radisson Blu Plaza** offers a superb selection of North and South Indian and Continental fare.

This itinerary ensures a day filled with artistic appreciation, historical insight, and luxurious relaxation, perfectly aligning with an "expensive," "relaxing," and "artsy" mood near Mysore.

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [10]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [12]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Mysore, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Mysore, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: 'b638c91a-d8fb-4e6a-8f0b-2718aca3699e'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text='I can only provide weather forecasts for locations within the United States. Mysore is not in the U.S.',
      thought_signature=b"\n\xae\x03\x01\xbe>\xf6\xfb/ Z\xdf<Hc\xfd\xa8\xfe:\x0bz\xfd\x0f\xbc\xd7\x15\xbc@*\xaa\x95V3e\x1a\xa3\x91;\x96\xfd\xc3\xbc\xfb\x02'\xe4I\xad>\x1cg\xde\xb6e\xba\xdd6\x7f[n\x8bI\xa1i\xba\\]\xf9\x8e\x07\x81\x95\xee\x88\x97*\xd3\xd1*\xd4\xf6\xa8>w\xd5\x1f\xcc\xca\xde\xa5u.\x812\xfb\\M...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=22,
  prompt_token_count=186,
  prom

I can only provide weather forecasts for locations within the United States. Mysore is not in the U.S.

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [13]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [14]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: '2e1acc7d-4e8f-49cf-84db-3b7302bdd9e5'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'top-rated hotels in San Francisco'
        },
        id='adk-c6c6a4cf-2ba1-4dba-ada2-7f6caf3a9c2f',
        name='call_db_agent'
      ),
      thought_signature=b"\n\x87\x05\x01\xbe>\xf6\xfb\xd5#\x87tp$x\xcc\xeb!\x13(\x8b\x80@\x00\xd7v'\xe6\xc0\x8b\x985\x1d\x05\x80\xbc\x9d\x8ak\xf2<\x1b\xba\xb0\x1a\xd5\xad\x80\xd4L\xc2\x89\x19\xe2\xa0\x9e\xc2\xafS.\x1a\x0cK\x02=\xac\xb5\xb9\xc9\xee\x03d\xe2\xecML\xe8<#5{\x83\xdb'\xe4\xda7\xda\xf9+'\xdc\x06\x08\x9c\x8f\xc6...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=N

An error occurred: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 23.618978314s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'quotaValue': '5'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '23s'}]}}

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [15]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [16]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 3cec202c-ac47-4797-9ea0-55aa0fe2bb00

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '3cec202c-ac47-4797-9ea0-55aa0fe2bb00'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! Lisbon is a fantastic choice for historic sites and delicious local food. Let's plan your 2-day adventure.

To start, for **Day 1**, how about we focus on some iconic historic areas and introduce you to some traditional Portuguese flavors?

Here’s a possible plan for Day 1:

**Day 1: Alfama's Charms & Culinary Delights**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & Lisbon Cathedral**
    *   Begin your day by wandering through the labyrinthine streets of Alfama, Lisbon's oldest district. Discover hidden courtyards, adm

Great! Lisbon is a fantastic choice for historic sites and delicious local food. Let's plan your 2-day adventure.

To start, for **Day 1**, how about we focus on some iconic historic areas and introduce you to some traditional Portuguese flavors?

Here’s a possible plan for Day 1:

**Day 1: Alfama's Charms & Culinary Delights**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & Lisbon Cathedral**
    *   Begin your day by wandering through the labyrinthine streets of Alfama, Lisbon's oldest district. Discover hidden courtyards, admire traditional tilework, and soak in the historic atmosphere.
    *   Visit the impressive Lisbon Cathedral (Sé de Lisboa), a national monument with a rich history dating back to the 12th century.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese meal at a local restaurant in Alfama. Try dishes like 'Bacalhau à Brás' (codfish with scrambled eggs and potatoes) or 'Arroz de Marisco' (seafood rice).
*   **Afternoon (2:30 PM - 6:00 PM): São Jorge Castle & Miradouro de Santa Luzia**
    *   Ascend to São Jorge Castle, a historic castle offering panoramic views over Lisbon. Explore its ramparts, gardens, and archaeological site.
    *   Walk to Miradouro de Santa Luzia for breathtaking views of the Alfama rooftops and the Tagus River.
*   **Evening (7:00 PM onwards): Fado Show with Dinner**
    *   Experience a traditional Fado show with dinner in Alfama or Bairro Alto. Fado is a soulful Portuguese musical genre, often performed in intimate settings.

How does this sound for your first day in Lisbon?

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '3cec202c-ac47-4797-9ea0-55aa0fe2bb00'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""You got it! I understand you're not a fan of castles. My apologies for including São Jorge Castle in the initial suggestion. Let's adjust **Day 1** to focus on other historical gems.

Here's the revised plan for Day 1, replacing the castle with a different historical experience:

**Day 1: Alfama's Charms & Baixa/Chiado History**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & Lisbon Cathedral**
    *   Begin your day by wandering through the labyrinthine streets of Alfama, Lisbon's oldest district. Discover hidden courtyards, admire traditional tilework, and so

You got it! I understand you're not a fan of castles. My apologies for including São Jorge Castle in the initial suggestion. Let's adjust **Day 1** to focus on other historical gems.

Here's the revised plan for Day 1, replacing the castle with a different historical experience:

**Day 1: Alfama's Charms & Baixa/Chiado History**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & Lisbon Cathedral**
    *   Begin your day by wandering through the labyrinthine streets of Alfama, Lisbon's oldest district. Discover hidden courtyards, admire traditional tilework, and soak in the historic atmosphere.
    *   Visit the impressive Lisbon Cathedral (Sé de Lisboa), a national monument with a rich history dating back to the 12th century.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese meal at a local restaurant in Alfama. Try dishes like 'Bacalhau à Brás' (codfish with scrambled eggs and potatoes) or 'Arroz de Marisco' (seafood rice).
*   **Afternoon (2:30 PM - 6:00 PM): Carmo Convent Ruins & Santa Justa Lift**
    *   Head to the historic Carmo Convent Ruins, a Gothic church largely destroyed in the 1755 earthquake. It now houses an archaeological museum and offers a poignant glimpse into Lisbon's past.
    *   Afterward, experience the Santa Justa Lift, an iconic iron elevator offering fantastic panoramic views over the Baixa district and connecting the lower streets with the higher Largo do Carmo.
*   **Evening (7:00 PM onwards): Fado Show with Dinner**
    *   Experience a traditional Fado show with dinner in Alfama or Bairro Alto. Fado is a soulful Portuguese musical genre, often performed in intimate settings.

How does this revised plan for Day 1 sound to you?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '3cec202c-ac47-4797-9ea0-55aa0fe2bb00'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excellent! I'm glad Day 1 is perfect. Let's move on to **Day 2**, keeping your interests in historic sites and great local food firmly in mind.

For Day 2, I suggest we explore the grand maritime history of Lisbon in Belém and then dive into a more contemporary food experience.

Here’s a possible plan for Day 2:

**Day 2: Belém's Maritime Grandeur & Culinary Explorations**

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime History**
    *   Travel to the historic Belém district, famous for its role in Portugal's Age of Discoveries.
    *   Visit the magnificent Jerónimos Monastery (Mosteiro dos Jerónimo

Excellent! I'm glad Day 1 is perfect. Let's move on to **Day 2**, keeping your interests in historic sites and great local food firmly in mind.

For Day 2, I suggest we explore the grand maritime history of Lisbon in Belém and then dive into a more contemporary food experience.

Here’s a possible plan for Day 2:

**Day 2: Belém's Maritime Grandeur & Culinary Explorations**

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime History**
    *   Travel to the historic Belém district, famous for its role in Portugal's Age of Discoveries.
    *   Visit the magnificent Jerónimos Monastery (Mosteiro dos Jerónimos), a UNESCO World Heritage site and a stunning example of Manueline architecture. Explore its church and cloisters.
    *   Walk along the waterfront to see the iconic Belém Tower (Torre de Belém) and the grand Monument to the Discoveries (Padrão dos Descobrimentos), both symbols of Portugal's rich naval past.
*   **Lunch (1:00 PM - 2:30 PM): Pastéis de Belém & Local Delights**
    *   Indulge in the world-famous *Pastéis de Belém* at the original factory, a truly essential culinary experience in Lisbon.
    *   Enjoy a light lunch at a local cafe in Belém, perhaps trying a traditional *Bifana* (pork sandwich) or *Caldo Verde* (green soup).
*   **Afternoon (2:30 PM - 6:00 PM): Mercado da Ribeira (Time Out Market) & Cais do Sodré**
    *   Head back towards central Lisbon and immerse yourself in Mercado da Ribeira, also known as Time Out Market. Explore the traditional market section with fresh produce and then enjoy the vibrant gourmet food hall, offering a vast array of Portuguese and international culinary delights from top chefs and restaurants.
    *   Take some time to explore the lively Cais do Sodré neighborhood surrounding the market.
*   **Evening (7:00 PM onwards): Dinner in Chiado or Bairro Alto**
    *   Enjoy dinner in the elegant Chiado district or the bustling Bairro Alto. These areas offer a wide range of restaurants, from traditional Portuguese cuisine to more contemporary dining options.
    *   If you're interested, Bairro Alto is also known for its lively bars and nightlife (optional).

How does this sound for your second day in Lisbon?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [17]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 55ab754d-c8f2-448a-8e65-f54e5e968917
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '55ab754d-c8f2-448a-8e65-f54e5e968917'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, fantastic! Lisbon is a wonderful choice for history and food lovers. We'll plan this step-by-step.

Here's a suggested itinerary for your first day in Lisbon:

### **Day 1: Exploring Historic Alfama**

*   **Morning (9:00 AM - 1:00 PM): Castelo de São Jorge & Lisbon Cathedral**
    *   Start your day by exploring **Castelo de São Jorge**, an ancient Moorish castle perched atop a hill. It offers incredible panoramic vi

Okay, fantastic! Lisbon is a wonderful choice for history and food lovers. We'll plan this step-by-step.

Here's a suggested itinerary for your first day in Lisbon:

### **Day 1: Exploring Historic Alfama**

*   **Morning (9:00 AM - 1:00 PM): Castelo de São Jorge & Lisbon Cathedral**
    *   Start your day by exploring **Castelo de São Jorge**, an ancient Moorish castle perched atop a hill. It offers incredible panoramic views of Lisbon and the Tagus River. Wander through its historic grounds and archaeological site.
    *   Afterward, make your way down to the **Lisbon Cathedral (Sé de Lisboa)**. This impressive 12th-century church is the city's oldest and showcases a blend of Romanesque, Gothic, and Baroque architecture. You can also visit its cloisters to see archaeological excavations.

*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Flavors in Alfama**
    *   Immerse yourself in the local culinary scene by finding a traditional *tasca* (small tavern) in the Alfama district for an authentic Portuguese lunch.
    *   **Food suggestions:** Try *Bacalhau à Brás* (a comforting dish of shredded salted cod with onions, potatoes, and scrambled eggs), *Caldo Verde* (a popular soup with shredded kale and chorizo), or a classic *Bifana* (a delicious marinated pork sandwich).

*   **Afternoon (2:30 PM - 6:00 PM): Alfama's Charm and Viewpoints**
    *   Spend your afternoon getting lost (in a good way!) in Alfama's labyrinthine narrow streets. It's a charming neighborhood filled with historic buildings and colorful houses.
    *   Be sure to stop at one of the many *miradouros* (viewpoints), such as **Miradouro de Santa Luzia** or **Miradouro das Portas do Sol**, for breathtaking vistas over Alfama's iconic red-tiled rooftops and the Tagus River.

*   **Evening (Optional): Fado Dinner Experience**
    *   As the birthplace of Fado music, Alfama is the perfect place to experience this soulful Portuguese tradition. Consider ending your day with a traditional Fado show accompanied by dinner at a local Fado house.

How does this sound for your first day in Lisbon? Are there any adjustments you'd like to make?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 1e204ea0-8a4c-4ff1-bcb0-8432784af702
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '1e204ea0-8a4c-4ff1-bcb0-8432784af702'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""I apologize, but it seems I've lost the context of our previous conversation. To plan Day 2 effectively, could you please remind me of:

*   **Your destination:**
*   **Total trip duration:**
*   **Your interests/preferences:**

Once I have this information, I'll be happy to craft a fantastic Day 2 itinerary for you!"""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata() partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_c

I apologize, but it seems I've lost the context of our previous conversation. To plan Day 2 effectively, could you please remind me of:

*   **Your destination:**
*   **Total trip duration:**
*   **Your interests/preferences:**

Once I have this information, I'll be happy to craft a fantastic Day 2 itinerary for you!

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
